# Chronological model comparison and hyperparameter tuning

## Purpose and safeguards

Compare a prevalence baseline, logistic regression fitted by SGD and histogram gradient boosting. Each nonbaseline family receives a six-candidate randomized search on three expanding temporal folds. Seed 42 fixes training-sample and search randomness.

The training cell runs only when final test metrics are absent; otherwise it preserves the existing completed run. The following cell displays the fold definitions, parameter spaces, feature ablation and warning log. Read these artifacts before interpreting the model comparison.

The 120,000-row tuning sample comes only from the training period. Final fitting uses 3,888,115 training records. Preprocessing is fitted inside each CV fold. Test data is not used to choose models, parameters or features.

Code-cell outputs are retained from the completed real-data run; this documentation update does not rerun or alter their computation.


In [1]:
from pathlib import Path
import sys, subprocess, json
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/schema.py').exists())
sys.path.insert(0, str(ROOT))
def run_stage(name):
    completed = subprocess.run([sys.executable, str(ROOT / 'scripts' / name)], cwd=ROOT, capture_output=True, text=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()
print('Project directory:', ROOT.name)


Project directory: flight_disruption_project


## Why the candidates differ

The baseline predicts prevalence for every flight and provides no schedule discrimination. Logistic SGD offers scalable regularized linear probabilities, with numerical scaling and one-hot categorical encoding. Histogram boosting captures nonlinear interactions with native categorical handling; unknown encoded categories become missing.

Temporal fold validation occurs in April, June and 1–29 August. A two-day gap separates each training end from validation start. Random internal early stopping is disabled.

The winner is selected by mean CV average precision. A fixed-parameter base-versus-extended feature comparison follows. The final selected model is then fitted on the full training period, and validation scores set a workload-based threshold before the test partition is loaded.


In [2]:
if not (ROOT / 'reports/test_metrics.json').exists():
    run_stage('train_models.py')
else:
    print('Keeping existing final evaluation. Do not repeatedly tune against the test set.')
display(pd.read_csv(ROOT / 'reports/model_comparison.csv'))

Keeping existing final evaluation. Do not repeatedly tune against the test set.


,model,cv_ap_before,cv_ap_after,cv_ap_std_after,parameters,validation_n,validation_prevalence,validation_average_precision,validation_roc_auc,validation_brier_score,validation_log_loss,validation_threshold,validation_alert_rate,validation_precision,validation_recall,validation_f1,validation_tn,validation_fp,validation_fn,validation_tp,validation_rank_budget,validation_precision_at_budget,validation_recall_at_budget,validation_lift_at_budget,validation_scenario_cost_per_1000
0,baseline,0.211544,0.211544,0.026349,{},923321,0.131669,0.131669,0.500000,0.122145,0.415141,0.220058,0.000000,0.000000,0.000000,0.000000,801748,0,121573,0,0.1,0.131699,0.100022,1.000223,658.346339
1,logistic_sgd,0.289323,0.302078,0.044297,"{""classifier__penalty"": ""elasticnet"", ""classif...",923321,0.131669,0.178538,0.598757,0.114525,0.387827,0.264213,0.099999,0.211814,0.160866,0.182858,728974,72774,102016,19557,0.1,0.211812,0.160866,1.608665,631.258251
2,hist_gradient_boosting,0.301569,0.307678,0.039368,"{""classifier__min_samples_leaf"": 100, ""classif...",923321,0.131669,0.186850,0.621499,0.117483,0.395051,0.317123,0.099544,0.213739,0.161590,0.184042,729482,72266,101928,19645,0.1,0.213902,0.162454,1.624540,630.231523


In [3]:
print(json.loads((ROOT / 'reports/cv_folds.json').read_text()))
print(json.loads((ROOT / 'reports/search_spaces.json').read_text()))
print(json.loads((ROOT / 'reports/feature_ablation.json').read_text()))
print(json.loads((ROOT / 'reports/training_warnings.json').read_text()))

[{'fold': 1, 'train_rows': 42330, 'validation_rows': 14929, 'train_start': '2015-01-01', 'train_end': '2015-03-29', 'validation_start': '2015-04-01', 'validation_end': '2015-04-30'}, {'fold': 2, 'train_rows': 72814, 'validation_rows': 15619, 'train_start': '2015-01-01', 'train_end': '2015-05-29', 'validation_start': '2015-06-01', 'validation_end': '2015-06-30'}, {'fold': 3, 'train_rows': 104329, 'validation_rows': 14681, 'train_start': '2015-01-01', 'train_end': '2015-07-29', 'validation_start': '2015-08-01', 'validation_end': '2015-08-29'}]
{'logistic_sgd': {'classifier__alpha': [1e-05, 0.0001, 0.001, 0.01], 'classifier__class_weight': [None, 'balanced'], 'classifier__penalty': ['l2', 'elasticnet']}, 'hist_gradient_boosting': {'classifier__learning_rate': [0.04, 0.08, 0.12], 'classifier__max_leaf_nodes': [15, 31, 63], 'classifier__l2_regularization': [0.0, 1.0, 10.0], 'classifier__min_samples_leaf': [30, 100]}}
{'model': 'hist_gradient_boosting', 'base_fold_ap': [0.2540628426948907, 0

## Recorded comparison

Mean CV AP changed from 0.289323 to 0.302078 for logistic SGD and from 0.301569 to 0.307678 for histogram boosting; baseline AP was 0.211544. The base-feature boosting variant subsequently achieved mean CV AP 0.308064 and was retained.

Selected boosting parameters are learning rate 0.04, maximum 15 leaves, minimum 100 samples per leaf and L2 regularization 1.0, with 150 boosting iterations. The recorded training-warning list is empty.

The comparison table's validation metrics describe the extended-feature candidates. Final base-model validation metrics are stored separately in `final_validation_metrics.json`. Do not combine their thresholds as though they refer to the same model. Continue to **05_evaluation.ipynb** for the frozen holdout results.
